## break the pdf into images and send it to gemini - flash - 2.5

In [ ]:
!pip install google-generativeai PyMuPDF pillow opencv-python numpy

In [15]:
import os
import sys
import base64
import json
import cv2
import numpy as np
from PIL import Image
import fitz  # PyMuPDF
import google.generativeai as genai
from typing import List, Tuple, Optional, Dict
import tempfile
import io
import shutil
from dotenv import load_dotenv
from pathlib import Path
import logging
from datetime import datetime
import glob

# Load environment variables from the .env file
load_dotenv('/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/.env')

# Add the path to import from prompt_store
sys.path.append('/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading')
from ocr.prompt_store import v_13_image_quality_2

# Configuration constants
MODEL_NAME = 'gemini-2.5-pro'
IMAGE_DIMENSION = 1536
PROMPT_VERSION = 'v_13_image_quality_2'

class ProcessingStats:
    """Class to track processing statistics"""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.total_pdfs = 0
        self.total_pages = 0
        self.total_images = 0
        self.successful_api_calls = 0
        self.failed_api_calls = 0
        self.json_files_created = 0
        self.failed_image_conversions = 0
        self.failed_json_saves = 0
        self.failed_images = []  # List of failed image file names
        self.errors = []
        self.start_time = datetime.now()
    
    def add_error(self, error_msg: str, file_name: str = ""):
        self.errors.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "file": file_name,
            "error": error_msg
        })
    
    def add_failed_image(self, image_name: str, error: str):
        self.failed_images.append({
            "image_name": image_name,
            "error": error,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
        self.failed_image_conversions += 1
    
    def get_summary(self):
        duration = datetime.now() - self.start_time
        return {
            "total_pdfs_processed": self.total_pdfs,
            "total_pages_processed": self.total_pages,
            "total_images_generated": self.total_images,
            "json_files_created": self.json_files_created,
            "successful_api_calls": self.successful_api_calls,
            "failed_api_calls": self.failed_api_calls,
            "failed_image_conversions": self.failed_image_conversions,
            "failed_json_saves": self.failed_json_saves,
            "total_errors": len(self.errors),
            "failed_images": self.failed_images,
            "processing_duration": str(duration),
            "model_used": MODEL_NAME,
            "prompt_version": PROMPT_VERSION,
            "image_dimension": IMAGE_DIMENSION,
            "errors": self.errors
        }

def setup_logger(log_file_path: str, subject: str) -> logging.Logger:
    """Set up a logger for a specific subject with improved formatting."""
    logger = logging.getLogger(f"{subject}_logger")
    logger.setLevel(logging.INFO)
    
    # Remove existing handlers to avoid duplication
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
    
    # Create file handler
    file_handler = logging.FileHandler(log_file_path, mode='w', encoding='utf-8')
    file_handler.setLevel(logging.INFO)
    
    # Create console handler
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    
    # Create improved formatter
    formatter = logging.Formatter('%(asctime)s | %(levelname)-7s | %(message)s', 
                                datefmt='%Y-%m-%d %H:%M:%S')
    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)
    
    # Add handlers to logger
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)
    
    return logger

def log_separator(logger: logging.Logger, title: str = "", char: str = "=", width: int = 80):
    """Create a formatted separator line in logs"""
    if title:
        title_formatted = f" {title} "
        padding = (width - len(title_formatted)) // 2
        separator = char * padding + title_formatted + char * padding
        if len(separator) < width:
            separator += char
    else:
        separator = char * width
    logger.info(separator)

def count_actual_files(subject_paths: Dict[str, Dict[str, str]]) -> Dict[str, Dict[str, int]]:
    """Count actual files in the output directories."""
    file_counts = {}
    
    for subject, paths in subject_paths.items():
        image_count = len(glob.glob(os.path.join(paths["images"], "*.jpeg")))
        json_count = len(glob.glob(os.path.join(paths["json"], "*.json")))
        pdf_count = len(glob.glob(os.path.join(paths["pdf"], "*.pdf")))
        
        file_counts[subject] = {
            "images": image_count,
            "json": json_count,
            "pdfs": pdf_count
        }
    
    return file_counts

def resize_image(image, dim=1536, save_path=None):
    """Resize image to specified dimension while maintaining aspect ratio."""
    if type(image) == str:
        image = Image.open(image)
    image1 = np.array(image.convert('RGB'))
    original_size = image1.shape
    image1 = image1.mean(axis=2)
    h, w = image1.shape
    
    if w > h:
        new_w = dim
        new_h = int(h * (dim / w))
    else:
        new_h = dim
        new_w = int(w * (dim / h))
        
    resized_image = cv2.resize(image1, (new_w, new_h), interpolation=cv2.INTER_AREA)
    resized_image_pil = Image.fromarray(resized_image)
    resized_image_pil = resized_image_pil.convert('RGB')
    
    if save_path:
        resized_image_pil.save(save_path)
        
    return original_size, (new_h, new_w), resized_image_pil

def pdf_to_images_with_naming(pdf_path: str, output_folder: str, pdf_name: str, logger: logging.Logger, stats: ProcessingStats, dim: int = 1536) -> int:
    """Convert PDF pages to images with specific naming convention."""
    try:
        os.makedirs(output_folder, exist_ok=True)
        pdf_document = fitz.open(pdf_path)
        num_pages = len(pdf_document)
        
        logger.info(f"📄 Converting PDF: {pdf_name}")
        logger.info(f"   └─ Pages: {num_pages} | Target dimension: {dim}px")
        
        for page_num in range(num_pages):
            try:
                page = pdf_document.load_page(page_num)
                mat = fitz.Matrix(2.0, 2.0)
                pix = page.get_pixmap(matrix=mat)
                
                img_data = pix.tobytes("ppm")
                pil_image = Image.open(io.BytesIO(img_data))
                
                original_size, new_size, resized_image = resize_image(pil_image, dim=dim)
                
                output_path = os.path.join(output_folder, f"{pdf_name}_page_{page_num + 1}_{dim}.jpeg")
                resized_image.save(output_path, "JPEG", quality=95)
                
                logger.info(f"   ✅ Page {page_num + 1}: {original_size[1]}x{original_size[0]} → {new_size[1]}x{new_size[0]}px")
                stats.total_images += 1
                stats.total_pages += 1
                
            except Exception as e:
                image_name = f"{pdf_name}_page_{page_num + 1}_{dim}.jpeg"
                error_msg = f"❌ Error processing page {page_num + 1} of {pdf_name}: {str(e)}"
                logger.error(error_msg)
                stats.add_error(error_msg, pdf_name)
                stats.add_failed_image(image_name, str(e))
        
        pdf_document.close()
        stats.total_pdfs += 1
        logger.info(f"✅ PDF conversion completed: {pdf_name} ({num_pages} pages)")
        return num_pages
        
    except Exception as e:
        error_msg = f"❌ Error opening PDF '{pdf_name}': {str(e)}"
        logger.error(error_msg)
        stats.add_error(error_msg, pdf_name)
        return 0

def send_single_image_to_gemini(image_path: str, prompt: str, api_key: str, logger: logging.Logger, stats: ProcessingStats) -> dict:
    """Send ONE image to Gemini in a single API call."""
    image_name = os.path.basename(image_path)
    
    try:
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel(MODEL_NAME)
        
        # Read and encode the single image
        with open(image_path, "rb") as f:
            image_data = f.read()
            image_size_mb = len(image_data) / (1024 * 1024)
            image_b64 = base64.b64encode(image_data).decode()
        
        # Prepare content for Gemini - ONE image only
        content = [
            prompt,
            {
                "mime_type": "image/jpeg",
                "data": image_b64
            }
        ]
        
        logger.info(f"🤖 Sending to {MODEL_NAME}: {image_name}")
        logger.info(f"   └─ Image size: {image_size_mb:.2f}MB | Prompt: {PROMPT_VERSION}")
        
        # Generate response
        response = model.generate_content(content)
        
        # Parse JSON response
        response_text = response.text.strip()
        
        if response_text.startswith("```json"):
            response_text = response_text[7:]
        if response_text.endswith("```"):
            response_text = response_text[:-3]
        
        json_response = json.loads(response_text)
        
        stats.successful_api_calls += 1
        logger.info(f"   ✅ Success: Response size: {len(response_text)} chars")
        
        return {"status": "success", "result": json_response, "image": image_name}
        
    except json.JSONDecodeError as e:
        error_msg = f"❌ JSON parsing error for {image_name}: {str(e)}"
        logger.error(error_msg)
        logger.error(f"   └─ Raw response: {response_text[:200]}..." if 'response_text' in locals() else "   └─ No response received")
        stats.add_failed_image(image_name, f"JSON parsing error: {str(e)}")
        stats.add_error(error_msg, image_name)
        stats.failed_api_calls += 1
        return {"status": "error", "error": str(e)}
    except Exception as e:
        error_msg = f"❌ API error for {image_name}: {str(e)}"
        logger.error(error_msg)
        stats.add_failed_image(image_name, f"API error: {str(e)}")
        stats.add_error(error_msg, image_name)
        stats.failed_api_calls += 1
        return {"status": "error", "error": str(e)}

def save_page_json(result: dict, pdf_name: str, page_num: int, image_name: str, json_folder: str, logger: logging.Logger, stats: ProcessingStats) -> None:
    """Save JSON result for a single page."""
    try:
        json_filename = f"{pdf_name}_page_{page_num}_{IMAGE_DIMENSION}.json"
        json_path = os.path.join(json_folder, json_filename)
        
        result_with_metadata = {
            "metadata": {
                "pdf_name": pdf_name,
                "page_number": page_num,
                "image_name": image_name,
                "image_dimension": IMAGE_DIMENSION,
                "model_used": MODEL_NAME,
                "prompt_version": PROMPT_VERSION,
                "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            },
            "result": result
        }
        
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(result_with_metadata, f, indent=2, ensure_ascii=False)
        
        stats.json_files_created += 1
        logger.info(f"💾 Saved JSON: {json_filename}")
        
    except Exception as e:
        error_msg = f"❌ Error saving JSON for {pdf_name} page {page_num}: {str(e)}"
        logger.error(error_msg)
        stats.add_error(error_msg, f"{pdf_name}_page_{page_num}")
        stats.failed_json_saves += 1

def create_output_structure(base_path: str) -> Dict[str, str]:
    """Create the organized folder structure for image_quality output."""
    image_quality_path = os.path.join(base_path, "image_quality")
    
    subjects = ["Physics", "Maths", "Chem","Bio"]
    subject_paths = {}
    
    for subject in subjects:
        subject_path = os.path.join(image_quality_path, subject)
        
        pdf_folder = os.path.join(subject_path, "pdf")
        images_folder = os.path.join(subject_path, "images")
        json_folder = os.path.join(subject_path, "json")
        
        os.makedirs(pdf_folder, exist_ok=True)
        os.makedirs(images_folder, exist_ok=True)
        os.makedirs(json_folder, exist_ok=True)
        
        subject_paths[subject] = {
            "pdf": pdf_folder,
            "images": images_folder,
            "json": json_folder,
            "base": subject_path
        }
        
        print(f"📁 Created structure for {subject}")
    
    return subject_paths

def get_pdf_paths() -> Dict[str, str]:
    """Get the paths to PDF folders for each subject."""
    base_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial"
    
    pdf_paths = {
        "Physics": os.path.join(base_path, "Physics", "Physics_Gemini", "phy"),
        "Maths": os.path.join(base_path, "maths", "maths_Gemini", "maths"),
        "Chem": os.path.join(base_path, "chemistry", "Chemistry_Gemini", "chem"),
        "Bio": os.path.join(base_path, "Bio", "bio_Gemini", "bio")
    }
    
    return pdf_paths

def process_subject_pdfs(subject: str, pdf_folder: str, output_paths: Dict[str, str], api_key: str) -> ProcessingStats:
    """Process all PDFs for a specific subject with individual page processing."""
    stats = ProcessingStats()
    log_file = os.path.join(output_paths["base"], "logs.txt")
    logger = setup_logger(log_file, subject)
    
    log_separator(logger, f"PROCESSING {subject.upper()}", "=")
    logger.info(f"🎯 Subject: {subject}")
    logger.info(f"📂 Source: {pdf_folder}")
    logger.info(f"🏭 Model: {MODEL_NAME}")
    logger.info(f"📋 Prompt: {PROMPT_VERSION}")
    logger.info(f"📐 Image Dimension: {IMAGE_DIMENSION}px")
    logger.info(f"📝 Processing Strategy: One API call per page")
    logger.info(f"💾 JSON Strategy: One file per page")
    logger.info(f"⏰ Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    log_separator(logger, "", "-")
    
    pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]
    
    if not pdf_files:
        logger.warning(f"⚠️  No PDF files found in {pdf_folder}")
        return stats
    
    logger.info(f"📊 Found {len(pdf_files)} PDF files to process")
    log_separator(logger, "", "-")
    
    prompt = v_13_image_quality_2
    
    for i, pdf_file in enumerate(pdf_files, 1):
        pdf_name = os.path.splitext(pdf_file)[0]
        pdf_path = os.path.join(pdf_folder, pdf_file)
        
        log_separator(logger, f"PDF {i}/{len(pdf_files)}: {pdf_file}", "-")
        
        try:
            # Step 1: Copy PDF to output folder
            shutil.copy2(pdf_path, output_paths["pdf"])
            logger.info(f"📋 Copied PDF to output folder")
            
            # Step 2: Convert PDF to images
            num_pages = pdf_to_images_with_naming(pdf_path, output_paths["images"], pdf_name, logger, stats, dim=IMAGE_DIMENSION)
            
            if num_pages == 0:
                logger.error(f"❌ Failed to convert PDF {pdf_file} to images")
                continue
            
            # Step 3: Process each image individually
            logger.info(f"🔄 Processing {num_pages} pages individually (one API call per page)")
            
            for page_num in range(1, num_pages + 1):
                image_name = f"{pdf_name}_page_{page_num}_{IMAGE_DIMENSION}.jpeg"
                image_path = os.path.join(output_paths["images"], image_name)
                
                if os.path.exists(image_path):
                    logger.info(f"📝 Processing page {page_num}/{num_pages}: {image_name}")
                    
                    # Send single image to Gemini
                    result = send_single_image_to_gemini(image_path, prompt, api_key, logger, stats)
                    
                    if result["status"] == "success":
                        # Save individual JSON for this page
                        save_page_json(result["result"], pdf_name, page_num, result["image"], output_paths["json"], logger, stats)
                    else:
                        logger.error(f"❌ Failed to process {image_name}: {result.get('error', 'Unknown error')}")
                else:
                    logger.warning(f"⚠️  Expected image not found: {image_name}")
                    stats.add_failed_image(image_name, "Image file not created")
            
            logger.info(f"✅ Completed: {pdf_file} ({num_pages} pages → {num_pages} JSON files)")
            
        except Exception as e:
            error_msg = f"💥 Unexpected error processing {pdf_file}: {str(e)}"
            logger.error(error_msg)
            stats.add_error(error_msg, pdf_file)
    
    # Log final statistics
    summary = stats.get_summary()
    log_separator(logger, f"{subject.upper()} SUMMARY", "=")
    logger.info(f"📊 PDFs processed: {summary['total_pdfs_processed']}")
    logger.info(f"📄 Pages processed: {summary['total_pages_processed']}")
    logger.info(f"🖼️  Images generated: {summary['total_images_generated']}")
    logger.info(f"💾 JSON files created: {summary['json_files_created']}")
    logger.info(f"✅ Successful API calls: {summary['successful_api_calls']}")
    logger.info(f"❌ Failed API calls: {summary['failed_api_calls']}")
    logger.info(f"🚫 Failed image conversions: {summary['failed_image_conversions']}")
    logger.info(f"🚫 Failed JSON saves: {summary['failed_json_saves']}")
    logger.info(f"⚠️  Total errors: {summary['total_errors']}")
    logger.info(f"⏱️  Duration: {summary['processing_duration']}")
    logger.info(f"🤖 Model used: {summary['model_used']}")
    logger.info(f"📋 Prompt version: {summary['prompt_version']}")
    
    # Log failed images details
    if summary['failed_images']:
        log_separator(logger, "FAILED IMAGES", "-")
        for failed_img in summary['failed_images']:
            logger.error(f"🚫 {failed_img['image_name']}: {failed_img['error']} [{failed_img['timestamp']}]")
    
    if summary['errors']:
        log_separator(logger, "ALL ERRORS", "-")
        for error in summary['errors']:
            logger.error(f"[{error['timestamp']}] {error['file']}: {error['error']}")
    
    log_separator(logger, f"END {subject.upper()}", "=")
    
    return stats

def main():
    """Main function to process all PDFs across all subjects with individual page processing."""
    api_key = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')
    if not api_key:
        raise ValueError("API key not found. Please set GEMINI_API_KEY in your .env file")
    
    print(f"🔑 Using API key: {api_key[:10]}...")
    print(f"🤖 Model: {MODEL_NAME}")
    print(f"📋 Prompt: {PROMPT_VERSION}")
    print(f"📐 Image dimension: {IMAGE_DIMENSION}px")
    print(f"📝 Processing Strategy: One API call per page")
    print(f"💾 JSON Strategy: One file per page")
    
    base_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial"
    
    print("\n📁 Creating output folder structure...")
    subject_paths = create_output_structure(base_path)
    
    pdf_paths = get_pdf_paths()
    
    overall_stats = {
        "Bio": None,
        "Physics": None,
        "Maths": None,
        "Chem": None
    }
    
    for subject in ["Bio", "Physics", "Maths", "Chem"]:
        if subject in pdf_paths and os.path.exists(pdf_paths[subject]):
            stats = process_subject_pdfs(subject, pdf_paths[subject], subject_paths[subject], api_key)
            overall_stats[subject] = stats.get_summary()
        else:
            print(f"⚠️  Warning: PDF folder not found for {subject}: {pdf_paths.get(subject, 'N/A')}")
    
    # Count actual files in directories
    print("\n🔍 Counting actual files in output directories...")
    actual_file_counts = count_actual_files(subject_paths)
    
    # Print overall summary with failures and actual file counts
    print(f"\n{'='*80}")
    print("🏆 OVERALL PROCESSING SUMMARY")
    print(f"{'='*80}")
    
    total_pdfs = 0
    total_pages = 0
    total_images = 0
    total_json_files = 0
    total_success = 0
    total_failures = 0
    total_errors = 0
    total_failed_images = 0
    total_failed_json = 0
    total_actual_images = 0
    total_actual_json = 0
    
    for subject, stats in overall_stats.items():
        actual_counts = actual_file_counts.get(subject, {"images": 0, "json": 0, "pdfs": 0})
        
        if stats:
            print(f"\n📚 {subject}:")
            print(f"   📄 PDFs: {stats['total_pdfs_processed']}")
            print(f"   📝 Pages: {stats['total_pages_processed']}")
            print(f"   🖼️  Images generated: {stats['total_images_generated']}")
            print(f"   💾 JSON files created: {stats['json_files_created']}")
            print(f"   ✅ Successful API calls: {stats['successful_api_calls']}")
            print(f"   ❌ Failed API calls: {stats['failed_api_calls']}")
            print(f"   🚫 Failed images: {stats['failed_image_conversions']}")
            print(f"   🚫 Failed JSON saves: {stats['failed_json_saves']}")
            print(f"   ⚠️  Total errors: {stats['total_errors']}")
            print(f"   ⏱️  Duration: {stats['processing_duration']}")
            print(f"   📁 ACTUAL FILES IN FOLDERS:")
            print(f"      🖼️  Images on disk: {actual_counts['images']}")
            print(f"      💾 JSON files on disk: {actual_counts['json']}")
            print(f"      📄 PDF files on disk: {actual_counts['pdfs']}")
            
            total_pdfs += stats['total_pdfs_processed']
            total_pages += stats['total_pages_processed']
            total_images += stats['total_images_generated']
            total_json_files += stats['json_files_created']
            total_success += stats['successful_api_calls']
            total_failures += stats['failed_api_calls']
            total_errors += stats['total_errors']
            total_failed_images += stats['failed_image_conversions']
            total_failed_json += stats['failed_json_saves']
            total_actual_images += actual_counts['images']
            total_actual_json += actual_counts['json']
        else:
            print(f"\n📚 {subject}: No data (folder not found or no PDFs)")
            print(f"   📁 ACTUAL FILES IN FOLDERS:")
            print(f"      🖼️  Images on disk: {actual_counts['images']}")
            print(f"      💾 JSON files on disk: {actual_counts['json']}")
            print(f"      📄 PDF files on disk: {actual_counts['pdfs']}")
            total_actual_images += actual_counts['images']
            total_actual_json += actual_counts['json']
    
    print(f"\n{'='*80}")
    print("📊 GRAND TOTALS:")
    print(f"   📄 Total PDFs processed: {total_pdfs}")
    print(f"   📝 Total Pages processed: {total_pages}")
    print(f"   🖼️  Total Images generated: {total_images}")
    print(f"   💾 Total JSON files created: {total_json_files}")
    print(f"   ✅ Total successful API calls: {total_success}")
    print(f"   ❌ Total failed API calls: {total_failures}")
    print(f"   🚫 Total failed image conversions: {total_failed_images}")
    print(f"   🚫 Total failed JSON saves: {total_failed_json}")
    print(f"   ⚠️  Total errors encountered: {total_errors}")
    print(f"\n📁 ACTUAL FILES ON DISK:")
    print(f"   🖼️  Total images files: {total_actual_images}")
    print(f"   💾 Total JSON files: {total_actual_json}")
    print(f"\n🤖 PROCESSING CONFIGURATION:")
    print(f"   Model used: {MODEL_NAME}")
    print(f"   Prompt version: {PROMPT_VERSION}")
    print(f"   Processing strategy: One API call per page")
    print(f"   Image dimension: {IMAGE_DIMENSION}px")
    print(f"{'='*80}")
    
    # Check for discrepancies
    if total_images != total_actual_images:
        print(f"\n⚠️  DISCREPANCY ALERT:")
        print(f"   Expected images: {total_images}")
        print(f"   Actual images on disk: {total_actual_images}")
        print(f"   Difference: {total_images - total_actual_images}")
    
    if total_json_files != total_actual_json:
        print(f"\n⚠️  JSON DISCREPANCY ALERT:")
        print(f"   Expected JSON files: {total_json_files}")
        print(f"   Actual JSON files on disk: {total_actual_json}")
        print(f"   Difference: {total_json_files - total_actual_json}")
    
    # Save enhanced summary to file
    enhanced_summary = {
        "processing_stats": overall_stats,
        "actual_file_counts": actual_file_counts,
        "totals": {
            "total_pdfs": total_pdfs,
            "total_pages": total_pages,
            "total_images_generated": total_images,
            "total_actual_images": total_actual_images,
            "total_json_created": total_json_files,
            "total_actual_json": total_actual_json,
            "total_successful_api_calls": total_success,
            "total_failed_api_calls": total_failures,
            "total_failed_images": total_failed_images,
            "total_failed_json": total_failed_json,
            "total_errors": total_errors
        },
        "configuration": {
            "model": MODEL_NAME,
            "prompt_version": PROMPT_VERSION,
            "processing_strategy": "one_api_call_per_page",
            "image_dimension": IMAGE_DIMENSION
        }
    }
    
    summary_path = os.path.join(base_path, "image_quality", "comprehensive_summary.json")
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(enhanced_summary, f, indent=2, ensure_ascii=False, default=str)
    
    print(f"\n💾 Comprehensive summary saved to: {summary_path}")
    print(f"📝 Individual logs saved in each subject's folder as 'logs.txt'")
    print(f"📁 Output structure created at: {os.path.join(base_path, 'image_quality')}")

if __name__ == "__main__":
    try:
        main()
    except ValueError as e:
        print(f"❌ Configuration Error: {e}")
    except Exception as e:
        print(f"💥 Error: {e}")
        import traceback
        traceback.print_exc()

2025-09-03 12:42:31 | INFO    | ================================ PROCESSING BIO ================================
2025-09-03 12:42:31 | INFO    | 🎯 Subject: Bio
2025-09-03 12:42:31 | INFO    | 📂 Source: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial/Bio/bio_Gemini/bio
2025-09-03 12:42:31 | INFO    | 🏭 Model: gemini-2.5-pro
2025-09-03 12:42:31 | INFO    | 📋 Prompt: v_13_image_quality_2
2025-09-03 12:42:31 | INFO    | 📐 Image Dimension: 1536px
2025-09-03 12:42:31 | INFO    | 📝 Processing Strategy: One API call per page
2025-09-03 12:42:31 | INFO    | 💾 JSON Strategy: One file per page
2025-09-03 12:42:31 | INFO    | ⏰ Started: 2025-09-03 12:42:31
2025-09-03 12:42:31 | INFO    | --------------------------------------------------------------------------------
2025-09-03 12:42:31 | INFO    | 📊 Found 8 PDF files to process
2025-09-03 12:42:31 | INFO    | -------------------------------------------------------------

🔑 Using API key: AIzaSyAO0X...
🤖 Model: gemini-2.5-pro
📋 Prompt: v_13_image_quality_2
📐 Image dimension: 1536px
📝 Processing Strategy: One API call per page
💾 JSON Strategy: One file per page

📁 Creating output folder structure...
📁 Created structure for Physics
📁 Created structure for Maths
📁 Created structure for Chem
📁 Created structure for Bio


2025-09-03 12:42:44 | INFO    |    ✅ Success: Response size: 273 chars
2025-09-03 12:42:44 | INFO    | 💾 Saved JSON: 05_1002113385815631141684559030_page_1_1536.json
2025-09-03 12:42:44 | INFO    | ✅ Completed: 05_1002113385815631141684559030.pdf (1 pages → 1 JSON files)
2025-09-03 12:42:44 | INFO    | ----------------- PDF 2/8: 03_1002102416961841141690700857.pdf -----------------
2025-09-03 12:42:44 | INFO    | 📋 Copied PDF to output folder
2025-09-03 12:42:44 | INFO    | 📄 Converting PDF: 03_1002102416961841141690700857
2025-09-03 12:42:44 | INFO    |    └─ Pages: 2 | Target dimension: 1536px
2025-09-03 12:42:44 | INFO    |    ✅ Page 1: 1191x1684 → 1086x1536px
2025-09-03 12:42:44 | INFO    |    ✅ Page 2: 1191x1684 → 1086x1536px
2025-09-03 12:42:44 | INFO    | ✅ PDF conversion completed: 03_1002102416961841141690700857 (2 pages)
2025-09-03 12:42:44 | INFO    | 🔄 Processing 2 pages individually (one API call per page)
2025-09-03 12:42:44 | INFO    | 📝 Processing page 1/2: 03_100210241


🔍 Counting actual files in output directories...

🏆 OVERALL PROCESSING SUMMARY

📚 Bio:
   📄 PDFs: 8
   📝 Pages: 28
   🖼️  Images generated: 28
   💾 JSON files created: 28
   ✅ Successful API calls: 28
   ❌ Failed API calls: 0
   🚫 Failed images: 0
   🚫 Failed JSON saves: 0
   ⚠️  Total errors: 0
   ⏱️  Duration: 0:06:20.383505
   📁 ACTUAL FILES IN FOLDERS:
      🖼️  Images on disk: 28
      💾 JSON files on disk: 28
      📄 PDF files on disk: 8

📚 Physics:
   📄 PDFs: 15
   📝 Pages: 62
   🖼️  Images generated: 62
   💾 JSON files created: 62
   ✅ Successful API calls: 62
   ❌ Failed API calls: 0
   🚫 Failed images: 0
   🚫 Failed JSON saves: 0
   ⚠️  Total errors: 0
   ⏱️  Duration: 0:14:17.105212
   📁 ACTUAL FILES IN FOLDERS:
      🖼️  Images on disk: 62
      💾 JSON files on disk: 62
      📄 PDF files on disk: 15

📚 Maths:
   📄 PDFs: 10
   📝 Pages: 138
   🖼️  Images generated: 138
   💾 JSON files created: 138
   ✅ Successful API calls: 138
   ❌ Failed API calls: 0
   🚫 Failed images: 0


## comparison 

In [ ]:
import os
import json
import hashlib
import shutil
from pathlib import Path

def get_json_result_hash(file_path):
    """Calculate hash of only the 'result' part of a JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Extract only the 'result' part
        if 'result' in data:
            result_data = data['result']
            # Convert to string and hash
            result_str = json.dumps(result_data, sort_keys=True)
            return hashlib.sha256(result_str.encode()).hexdigest(), result_data
        else:
            return "NO_RESULT_KEY", None
    except Exception as e:
        return f"ERROR: {str(e)}", None

def find_and_compare_result_differences(result1, result2):
    """Find differences between two result dictionaries."""
    differences = {}
    
    # Get all unique keys from both results
    all_keys = set(result1.keys()) | set(result2.keys())
    
    for key in all_keys:
        val1 = result1.get(key, "MISSING")
        val2 = result2.get(key, "MISSING")
        
        if val1 != val2:
            differences[key] = {
                "flash_2_5": val1,
                "pro_2_5": val2
            }
    
    return differences

def copy_corresponding_image(json_file_path, source_folder, dest_folder):
    """Copy the corresponding image file from source to destination."""
    try:
        # Convert JSON path to image path
        json_name = os.path.basename(json_file_path)
        image_name = json_name.replace('.json', '.jpeg')
        
        # Extract the relative path structure
        rel_path = os.path.relpath(json_file_path, os.path.join(source_folder, 'json'))
        rel_dir = os.path.dirname(rel_path)
        
        # Source and destination image paths
        source_image_path = os.path.join(source_folder, 'images', rel_dir, image_name)
        dest_image_dir = os.path.join(dest_folder, 'images', rel_dir)
        dest_image_path = os.path.join(dest_image_dir, image_name)
        
        # Create destination directory if it doesn't exist
        os.makedirs(dest_image_dir, exist_ok=True)
        
        # Copy the image if it exists
        if os.path.exists(source_image_path):
            shutil.copy2(source_image_path, dest_image_path)
            return True, f"Copied: {image_name}"
        else:
            return False, f"Image not found: {source_image_path}"
            
    except Exception as e:
        return False, f"Error copying image: {str(e)}"

def compare_folders():
    """Compare JSON files between two image_quality folders and copy mismatched images."""
    
    # Define the folder paths
    folder1 = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial/image_quality_flash_2.5"
    folder2 = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial/image_quality_pro_2.5"
    differences_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial/differences"
    
    # Create differences folder if it doesn't exist
    os.makedirs(differences_folder, exist_ok=True)
    
    # Statistics
    total_files_compared = 0
    files_with_differences = 0
    files_only_in_flash = 0
    files_only_in_pro = 0
    images_copied_flash = 0
    images_copied_pro = 0
    image_copy_errors = []
    
    # Get all JSON files from both folders (excluding logs.txt and comprehensive_summary.json)
    def get_json_files(folder):
        json_files = {}
        if os.path.exists(folder):
            for root, dirs, files in os.walk(folder):
                for file in files:
                    if (file.endswith('.json') and 
                        file not in ['logs.txt', 'comprehensive_summary.json']):
                        
                        full_path = os.path.join(root, file)
                        # Create relative path from the base folder
                        rel_path = os.path.relpath(full_path, folder)
                        json_files[rel_path] = full_path
        return json_files
    
    print("🔍 SCANNING FOLDERS...")
    print(f"Flash 2.5 folder: {folder1}")
    print(f"Pro 2.5 folder: {folder2}")
    print(f"Differences folder: {differences_folder}")
    print("=" * 80)
    
    flash_files = get_json_files(folder1)
    pro_files = get_json_files(folder2)
    
    print(f"📊 FOUND FILES:")
    print(f"   Flash 2.5: {len(flash_files)} JSON files")
    print(f"   Pro 2.5: {len(pro_files)} JSON files")
    print("=" * 80)
    
    # Find common files and files unique to each folder
    common_files = set(flash_files.keys()) & set(pro_files.keys())
    only_flash = set(flash_files.keys()) - set(pro_files.keys())
    only_pro = set(pro_files.keys()) - set(flash_files.keys())
    
    files_only_in_flash = len(only_flash)
    files_only_in_pro = len(only_pro)
    
    print(f"📋 FILE ANALYSIS:")
    print(f"   Common files: {len(common_files)}")
    print(f"   Only in Flash 2.5: {files_only_in_flash}")
    print(f"   Only in Pro 2.5: {files_only_in_pro}")
    print("=" * 80)
    
    if only_flash:
        print("📁 FILES ONLY IN FLASH 2.5:")
        for file in list(only_flash)[:5]:  # Show first 5
            print(f"   • {file}")
        if len(only_flash) > 5:
            print(f"   ... and {len(only_flash) - 5} more")
        print()
    
    if only_pro:
        print("📁 FILES ONLY IN PRO 2.5:")
        for file in list(only_pro)[:5]:  # Show first 5
            print(f"   • {file}")
        if len(only_pro) > 5:
            print(f"   ... and {len(only_pro) - 5} more")
        print()
    
    print("🔍 COMPARING COMMON FILES...")
    print("=" * 80)
    
    # Compare common files
    for rel_path in common_files:
        total_files_compared += 1
        
        flash_file = flash_files[rel_path]
        pro_file = pro_files[rel_path]
        
        # Get result hashes and data
        flash_hash, flash_result = get_json_result_hash(flash_file)
        pro_hash, pro_result = get_json_result_hash(pro_file)
        
        if flash_hash != pro_hash and flash_result and pro_result:
            files_with_differences += 1
            
            # Find specific differences
            differences = find_and_compare_result_differences(flash_result, pro_result)
            
            print(f"📄 DIFFERENCE FOUND: {rel_path}")
            print(f"   Differences: {len(differences)}")
            
            # Create the difference JSON file
            diff_json_path = os.path.join(differences_folder, rel_path)
            diff_json_dir = os.path.dirname(diff_json_path)
            os.makedirs(diff_json_dir, exist_ok=True)
            
            # Create difference data (without full_results)
            diff_data = {
                "file_info": {
                    "relative_path": rel_path,
                    "flash_2_5_path": flash_file,
                    "pro_2_5_path": pro_file
                },
                "differences_found": len(differences),
                "differences": differences
            }
            
            # Save difference JSON
            with open(diff_json_path, 'w', encoding='utf-8') as f:
                json.dump(diff_data, f, indent=2, ensure_ascii=False)
            
            # Copy corresponding images from both folders
            # Copy from Flash 2.5 folder
            success_flash, msg_flash = copy_corresponding_image(
                flash_file, 
                os.path.dirname(os.path.dirname(flash_file)),  # Go up to get base folder
                os.path.dirname(os.path.dirname(diff_json_path))  # Go up to get base differences folder
            )
            
            if success_flash:
                images_copied_flash += 1
                print(f"   ✅ {msg_flash} (Flash 2.5)")
            else:
                image_copy_errors.append(f"Flash 2.5 - {rel_path}: {msg_flash}")
                print(f"   ❌ {msg_flash} (Flash 2.5)")
            
            # Copy from Pro 2.5 folder
            success_pro, msg_pro = copy_corresponding_image(
                pro_file,
                os.path.dirname(os.path.dirname(pro_file)),  # Go up to get base folder
                os.path.dirname(os.path.dirname(diff_json_path))  # Go up to get base differences folder
            )
            
            if success_pro:
                images_copied_pro += 1
                print(f"   ✅ {msg_pro} (Pro 2.5)")
            else:
                image_copy_errors.append(f"Pro 2.5 - {rel_path}: {msg_pro}")
                print(f"   ❌ {msg_pro} (Pro 2.5)")
            
            print()
    
    # Print final summary
    print("=" * 80)
    print("📊 FINAL SUMMARY")
    print("=" * 80)
    print(f"📁 Total JSON files compared: {total_files_compared}")
    print(f"📄 Files with differences: {files_with_differences}")
    print(f"📄 Files only in Flash 2.5: {files_only_in_flash}")
    print(f"📄 Files only in Pro 2.5: {files_only_in_pro}")
    print()
    print(f"🖼️  IMAGES COPIED:")
    print(f"   Flash 2.5 images: {images_copied_flash}")
    print(f"   Pro 2.5 images: {images_copied_pro}")
    print(f"   Total images copied: {images_copied_flash + images_copied_pro}")
    
    if image_copy_errors:
        print(f"\n❌ IMAGE COPY ERRORS ({len(image_copy_errors)}):")
        for error in image_copy_errors[:10]:  # Show first 10 errors
            print(f"   • {error}")
        if len(image_copy_errors) > 10:
            print(f"   ... and {len(image_copy_errors) - 10} more errors")
    
    print("\n✅ COMPARISON COMPLETE!")
    print(f"📁 Results saved in: {differences_folder}")

# Run the comparison
if __name__ == "__main__":
    compare_folders()

In [1]:
import os
import json
import hashlib
import shutil
from pathlib import Path

def get_json_result_hash(file_path):
    """Calculate hash of only the 'result' part of a JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Extract only the 'result' part
        if 'result' in data:
            result_data = data['result']
            # Convert to string and hash
            result_str = json.dumps(result_data, sort_keys=True)
            return hashlib.sha256(result_str.encode()).hexdigest(), result_data
        else:
            return "NO_RESULT_KEY", None
    except Exception as e:
        return f"ERROR: {str(e)}", None

def copy_corresponding_image(json_file_path, source_folder, dest_folder):
    """Copy the corresponding image file from source to destination."""
    try:
        # Convert JSON path to image path
        json_name = os.path.basename(json_file_path)
        image_name = json_name.replace('.json', '.jpeg')
        
        # Extract the relative path structure
        rel_path = os.path.relpath(json_file_path, os.path.join(source_folder, 'json'))
        rel_dir = os.path.dirname(rel_path)
        
        # Source and destination image paths
        source_image_path = os.path.join(source_folder, 'images', rel_dir, image_name)
        dest_image_dir = os.path.join(dest_folder, 'images', rel_dir)
        dest_image_path = os.path.join(dest_image_dir, image_name)
        
        # Create destination directory if it doesn't exist
        os.makedirs(dest_image_dir, exist_ok=True)
        
        # Copy the image if it exists
        if os.path.exists(source_image_path):
            shutil.copy2(source_image_path, dest_image_path)
            return True, f"Copied: {image_name}"
        else:
            return False, f"Image not found: {source_image_path}"
            
    except Exception as e:
        return False, f"Error copying image: {str(e)}"

def compare_folders():
    """Compare JSON files between two image_quality folders and copy files with same results."""
    
    # Define the folder paths
    folder1 = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial_1/image_quality_flash_2.5"
    folder2 = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial_1/image_quality_pro_2.5"
    same_results_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial_1/same_results"
    
    # Create same_results folder if it doesn't exist
    os.makedirs(same_results_folder, exist_ok=True)
    
    # Statistics
    total_files_compared = 0
    files_with_same_results = 0
    files_only_in_flash = 0
    files_only_in_pro = 0
    images_copied_flash = 0
    images_copied_pro = 0
    image_copy_errors = []
    
    # Get all JSON files from both folders (excluding logs.txt and comprehensive_summary.json)
    def get_json_files(folder):
        json_files = {}
        if os.path.exists(folder):
            for root, dirs, files in os.walk(folder):
                for file in files:
                    if (file.endswith('.json') and 
                        file not in ['logs.txt', 'comprehensive_summary.json']):
                        
                        full_path = os.path.join(root, file)
                        # Create relative path from the base folder
                        rel_path = os.path.relpath(full_path, folder)
                        json_files[rel_path] = full_path
        return json_files
    
    print("🔍 SCANNING FOLDERS...")
    print(f"Flash 2.5 folder: {folder1}")
    print(f"Pro 2.5 folder: {folder2}")
    print(f"Same results folder: {same_results_folder}")
    print("=" * 80)
    
    flash_files = get_json_files(folder1)
    pro_files = get_json_files(folder2)
    
    print(f"📊 FOUND FILES:")
    print(f"   Flash 2.5: {len(flash_files)} JSON files")
    print(f"   Pro 2.5: {len(pro_files)} JSON files")
    print("=" * 80)
    
    # Find common files and files unique to each folder
    common_files = set(flash_files.keys()) & set(pro_files.keys())
    only_flash = set(flash_files.keys()) - set(pro_files.keys())
    only_pro = set(pro_files.keys()) - set(flash_files.keys())
    
    files_only_in_flash = len(only_flash)
    files_only_in_pro = len(only_pro)
    
    print(f"📋 FILE ANALYSIS:")
    print(f"   Common files: {len(common_files)}")
    print(f"   Only in Flash 2.5: {files_only_in_flash}")
    print(f"   Only in Pro 2.5: {files_only_in_pro}")
    print("=" * 80)
    
    if only_flash:
        print("📁 FILES ONLY IN FLASH 2.5:")
        for file in list(only_flash)[:5]:  # Show first 5
            print(f"   • {file}")
        if len(only_flash) > 5:
            print(f"   ... and {len(only_flash) - 5} more")
        print()
    
    if only_pro:
        print("📁 FILES ONLY IN PRO 2.5:")
        for file in list(only_pro)[:5]:  # Show first 5
            print(f"   • {file}")
        if len(only_pro) > 5:
            print(f"   ... and {len(only_pro) - 5} more")
        print()
    
    print("🔍 COMPARING COMMON FILES FOR SAME RESULTS...")
    print("=" * 80)
    
    # Compare common files for SAME results
    for rel_path in common_files:
        total_files_compared += 1
        
        flash_file = flash_files[rel_path]
        pro_file = pro_files[rel_path]
        
        # Get result hashes and data
        flash_hash, flash_result = get_json_result_hash(flash_file)
        pro_hash, pro_result = get_json_result_hash(pro_file)
        
        # Check if results are SAME (not different)
        if flash_hash == pro_hash and flash_result and pro_result and flash_hash != "NO_RESULT_KEY" and flash_hash != "ERROR":
            files_with_same_results += 1
            
            print(f"📄 SAME RESULTS FOUND: {rel_path}")
            print(f"   Hash: {flash_hash[:16]}...")
            
            # Create the same results JSON file
            same_json_path = os.path.join(same_results_folder, rel_path)
            same_json_dir = os.path.dirname(same_json_path)
            os.makedirs(same_json_dir, exist_ok=True)
            
            # Create same results data
            same_data = {
                "file_info": {
                    "relative_path": rel_path,
                    "flash_2_5_path": flash_file,
                    "pro_2_5_path": pro_file
                },
                "results_match": True,
                "result_hash": flash_hash,
                "matching_result": flash_result
            }
            
            # Save same results JSON
            with open(same_json_path, 'w', encoding='utf-8') as f:
                json.dump(same_data, f, indent=2, ensure_ascii=False)
            
            # Copy corresponding images from both folders
            # Copy from Flash 2.5 folder
            success_flash, msg_flash = copy_corresponding_image(
                flash_file, 
                os.path.dirname(os.path.dirname(flash_file)),  # Go up to get base folder
                os.path.dirname(os.path.dirname(same_json_path))  # Go up to get base same_results folder
            )
            
            if success_flash:
                images_copied_flash += 1
                print(f"   ✅ {msg_flash} (Flash 2.5)")
            else:
                image_copy_errors.append(f"Flash 2.5 - {rel_path}: {msg_flash}")
                print(f"   ❌ {msg_flash} (Flash 2.5)")
            
            # Copy from Pro 2.5 folder
            success_pro, msg_pro = copy_corresponding_image(
                pro_file,
                os.path.dirname(os.path.dirname(pro_file)),  # Go up to get base folder
                os.path.dirname(os.path.dirname(same_json_path))  # Go up to get base same_results folder
            )
            
            if success_pro:
                images_copied_pro += 1
                print(f"   ✅ {msg_pro} (Pro 2.5)")
            else:
                image_copy_errors.append(f"Pro 2.5 - {rel_path}: {msg_pro}")
                print(f"   ❌ {msg_pro} (Pro 2.5)")
            
            print()
    
    # Print final summary
    print("=" * 80)
    print("📊 FINAL SUMMARY")
    print("=" * 80)
    print(f"📁 Total JSON files compared: {total_files_compared}")
    print(f"📄 Files with SAME results: {files_with_same_results}")
    print(f"📄 Files with DIFFERENT results: {total_files_compared - files_with_same_results}")
    print(f"📄 Files only in Flash 2.5: {files_only_in_flash}")
    print(f"📄 Files only in Pro 2.5: {files_only_in_pro}")
    print()
    print(f"🖼️  IMAGES COPIED:")
    print(f"   Flash 2.5 images: {images_copied_flash}")
    print(f"   Pro 2.5 images: {images_copied_pro}")
    print(f"   Total images copied: {images_copied_flash + images_copied_pro}")
    
    if image_copy_errors:
        print(f"\n❌ IMAGE COPY ERRORS ({len(image_copy_errors)}):")
        for error in image_copy_errors[:10]:  # Show first 10 errors
            print(f"   • {error}")
        if len(image_copy_errors) > 10:
            print(f"   ... and {len(image_copy_errors) - 10} more errors")
    
    print("\n✅ COMPARISON COMPLETE!")
    print(f"📁 Results saved in: {same_results_folder}")

# Run the comparison
if __name__ == "__main__":
    compare_folders()

🔍 SCANNING FOLDERS...
Flash 2.5 folder: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial_1/image_quality_flash_2.5
Pro 2.5 folder: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial_1/image_quality_pro_2.5
Same results folder: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial_1/same_results
📊 FOUND FILES:
   Flash 2.5: 0 JSON files
   Pro 2.5: 0 JSON files
📋 FILE ANALYSIS:
   Common files: 0
   Only in Flash 2.5: 0
   Only in Pro 2.5: 0
🔍 COMPARING COMMON FILES FOR SAME RESULTS...
📊 FINAL SUMMARY
📁 Total JSON files compared: 0
📄 Files with SAME results: 0
📄 Files with DIFFERENT results: 0
📄 Files only in Flash 2.5: 0
📄 Files only in Pro 2.5: 0

🖼️  IMAGES COPIED:
   Flash 2.5 images: 0
   Pro 2.5 images: 0
   Total images copied: 0

✅ COMPARISON COMPLETE!
📁 Resu

In [2]:
import os
import glob
from pathlib import Path

def count_files_in_folder():
    """Count images and JSON files in image_quality_pro_2.5 folder and its subfolders."""
    
    base_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial/image_quality_pro_2.5"
    
    print(f"📊 Analyzing folder: {base_path}")
    print(f"{'='*80}")
    
    if not os.path.exists(base_path):
        print(f"❌ Folder does not exist: {base_path}")
        return
    
    # Initialize counters
    total_images = 0
    total_json = 0
    total_pdfs = 0
    total_other = 0
    
    # Dictionary to store counts per subject
    subject_counts = {}
    
    # Get all immediate subdirectories (subjects)
    subjects = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
    subjects.sort()
    
    print(f"📚 Found {len(subjects)} subject folders: {', '.join(subjects)}")
    print(f"{'='*80}")
    
    for subject in subjects:
        subject_path = os.path.join(base_path, subject)
        
        # Initialize subject counters
        subject_images = 0
        subject_json = 0
        subject_pdfs = 0
        subject_other = 0
        
        print(f"\n📁 Subject: {subject}")
        print(f"{'-'*50}")
        
        # Get subfolders within each subject (pdf, images, json)
        subfolders = [d for d in os.listdir(subject_path) if os.path.isdir(os.path.join(subject_path, d))]
        subfolders.sort()
        
        # Analyze each subfolder
        for subfolder in subfolders:
            subfolder_path = os.path.join(subject_path, subfolder)
            
            # Count different file types
            images = len(glob.glob(os.path.join(subfolder_path, "*.jpeg"))) + \
                    len(glob.glob(os.path.join(subfolder_path, "*.jpg"))) + \
                    len(glob.glob(os.path.join(subfolder_path, "*.png")))
            
            json_files = len(glob.glob(os.path.join(subfolder_path, "*.json")))
            
            pdf_files = len(glob.glob(os.path.join(subfolder_path, "*.pdf")))
            
            # Count other files (excluding logs.txt and summary files)
            all_files = []
            for ext in ['*']:
                all_files.extend(glob.glob(os.path.join(subfolder_path, f"*.{ext}" if ext != '*' else '*')))
            
            # Filter out known file types and unwanted files
            other_files = []
            for file_path in all_files:
                filename = os.path.basename(file_path)
                if (not filename.endswith(('.jpeg', '.jpg', '.png', '.json', '.pdf')) and 
                    filename not in ['logs.txt', 'comprehensive_summary.json'] and
                    os.path.isfile(file_path)):
                    other_files.append(file_path)
            
            other_count = len(other_files)
            
            # Update subject counters
            subject_images += images
            subject_json += json_files
            subject_pdfs += pdf_files
            subject_other += other_count
            
            # Print subfolder details
            print(f"   📂 {subfolder}/")
            print(f"      🖼️  Images: {images}")
            print(f"      💾 JSON files: {json_files}")
            print(f"      📄 PDF files: {pdf_files}")
            if other_count > 0:
                print(f"      📋 Other files: {other_count}")
                for other_file in other_files:
                    print(f"         - {os.path.basename(other_file)}")
        
        # Store subject totals
        subject_counts[subject] = {
            "images": subject_images,
            "json": subject_json,
            "pdfs": subject_pdfs,
            "other": subject_other,
            "total": subject_images + subject_json + subject_pdfs + subject_other
        }
        
        # Update grand totals
        total_images += subject_images
        total_json += subject_json
        total_pdfs += subject_pdfs
        total_other += subject_other
        
        # Print subject summary
        print(f"\n   📊 {subject} TOTALS:")
        print(f"      🖼️  Total Images: {subject_images}")
        print(f"      💾 Total JSON: {subject_json}")
        print(f"      📄 Total PDFs: {subject_pdfs}")
        if subject_other > 0:
            print(f"      📋 Total Other: {subject_other}")
        print(f"      📈 Total Files: {subject_images + subject_json + subject_pdfs + subject_other}")
    
    # Print overall summary
    print(f"\n{'='*80}")
    print(f"🏆 OVERALL SUMMARY FOR image_quality_pro_2.5")
    print(f"{'='*80}")
    
    print(f"\n📊 BY SUBJECT:")
    for subject, counts in subject_counts.items():
        print(f"   📚 {subject}:")
        print(f"      🖼️  Images: {counts['images']}")
        print(f"      💾 JSON: {counts['json']}")
        print(f"      📄 PDFs: {counts['pdfs']}")
        if counts['other'] > 0:
            print(f"      📋 Other: {counts['other']}")
        print(f"      📈 Total: {counts['total']}")
    
    print(f"\n📊 GRAND TOTALS:")
    print(f"   🖼️  Total Images: {total_images}")
    print(f"   💾 Total JSON files: {total_json}")
    print(f"   📄 Total PDF files: {total_pdfs}")
    if total_other > 0:
        print(f"   📋 Total Other files: {total_other}")
    print(f"   📈 TOTAL FILES: {total_images + total_json + total_pdfs + total_other}")
    
    print(f"\n📊 BREAKDOWN BY FILE TYPE:")
    print(f"   🖼️  Images: {total_images} ({total_images/(total_images + total_json + total_pdfs + total_other)*100:.1f}%)")
    print(f"   💾 JSON: {total_json} ({total_json/(total_images + total_json + total_pdfs + total_other)*100:.1f}%)")
    print(f"   📄 PDFs: {total_pdfs} ({total_pdfs/(total_images + total_json + total_pdfs + total_other)*100:.1f}%)")
    if total_other > 0:
        print(f"   📋 Other: {total_other} ({total_other/(total_images + total_json + total_pdfs + total_other)*100:.1f}%)")
    
    # Check for expected patterns
    print(f"\n🔍 ANALYSIS:")
    if total_images == total_json:
        print(f"   ✅ Perfect match: {total_images} images = {total_json} JSON files")
    else:
        print(f"   ⚠️  Mismatch: {total_images} images ≠ {total_json} JSON files")
        print(f"      Difference: {abs(total_images - total_json)} files")
    
    # Check subject balance
    if len(subject_counts) > 1:
        image_counts = [counts['images'] for counts in subject_counts.values()]
        json_counts = [counts['json'] for counts in subject_counts.values()]
        
        if len(set(image_counts)) == 1:
            print(f"   ✅ All subjects have equal image counts: {image_counts[0]} each")
        else:
            print(f"   📊 Image distribution varies by subject:")
            for subject, counts in subject_counts.items():
                print(f"      {subject}: {counts['images']} images")
        
        if len(set(json_counts)) == 1:
            print(f"   ✅ All subjects have equal JSON counts: {json_counts[0]} each")
        else:
            print(f"   📊 JSON distribution varies by subject:")
            for subject, counts in subject_counts.items():
                print(f"      {subject}: {counts['json']} JSON files")
    
    print(f"{'='*80}")
    
    return {
        "total_images": total_images,
        "total_json": total_json,
        "total_pdfs": total_pdfs,
        "total_other": total_other,
        "subject_counts": subject_counts,
        "subjects": subjects
    }

# Run the analysis
if __name__ == "__main__":
    results = count_files_in_folder()

📊 Analyzing folder: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial/image_quality_pro_2.5
❌ Folder does not exist: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial/image_quality_pro_2.5


In [3]:
import os
import json
import re
from collections import defaultdict
from PIL import Image
import glob

def extract_base_filename(filename):
    """Extract base filename without page number and extension"""
    # Remove extension first
    name_without_ext = os.path.splitext(filename)[0]
    # Remove page number pattern (e.g., _page_1_1536)
    base_name = re.sub(r'_page_\d+_\d+$', '', name_without_ext)
    return base_name

def club_json_files(json_dir, output_dir):
    """Club JSON files with same base filename into single files"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Group files by base filename
    file_groups = defaultdict(list)
    
    for filename in os.listdir(json_dir):
        if filename.endswith('.json'):
            base_name = extract_base_filename(filename)
            file_groups[base_name].append(filename)
    
    # Process each group
    for base_name, filenames in file_groups.items():
        combined_data = {
            "base_filename": base_name,
            "pages": []
        }
        
        # Sort filenames by page number
        filenames.sort(key=lambda x: int(re.search(r'_page_(\d+)_', x).group(1)))
        
        for filename in filenames:
            file_path = os.path.join(json_dir, filename)
            with open(file_path, 'r') as f:
                page_data = json.load(f)
            
            # Extract page number
            page_match = re.search(r'_page_(\d+)_', filename)
            page_num = int(page_match.group(1)) if page_match else 1
            
            combined_data["pages"].append({
                "page_number": page_num,
                "original_filename": filename,
                "data": page_data
            })
        
        # Save combined JSON
        output_filename = f"{base_name}_combined.json"
        output_path = os.path.join(output_dir, output_filename)
        
        with open(output_path, 'w') as f:
            json.dump(combined_data, f, indent=2)
        
        print(f"Created: {output_filename} with {len(filenames)} pages")

def images_to_pdf(images_dir, output_dir):
    """Convert grouped images to PDFs"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Group images by base filename
    image_groups = defaultdict(list)
    
    for filename in os.listdir(images_dir):
        if filename.lower().endswith(('.jpeg', '.jpg', '.png')):
            base_name = extract_base_filename(filename)
            image_groups[base_name].append(filename)
    
    # Process each group
    for base_name, filenames in image_groups.items():
        # Sort filenames by page number
        filenames.sort(key=lambda x: int(re.search(r'_page_(\d+)_', x).group(1)))
        
        images = []
        for filename in filenames:
            img_path = os.path.join(images_dir, filename)
            img = Image.open(img_path)
            # Convert to RGB if necessary
            if img.mode != 'RGB':
                img = img.convert('RGB')
            images.append(img)
        
        if images:
            output_filename = f"{base_name}_combined.pdf"
            output_path = os.path.join(output_dir, output_filename)
            
            # Save as PDF
            images[0].save(output_path, save_all=True, append_images=images[1:])
            print(f"Created: {output_filename} with {len(filenames)} pages")

def main():
    # Define paths
    base_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial_1/same_results/Physics"
    json_dir = os.path.join(base_dir, "json")
    images_dir = os.path.join(base_dir, "images")
    
    # Create output directories
    json_output_dir = os.path.join(base_dir, "phy_final_json")
    pdf_output_dir = os.path.join(base_dir, "phy_final_pdf")
    
    print("Starting JSON file clubbing...")
    club_json_files(json_dir, json_output_dir)
    
    print("\nStarting image to PDF conversion...")
    images_to_pdf(images_dir, pdf_output_dir)
    
    print("\nProcess completed!")
    print(f"Combined JSON files saved in: {json_output_dir}")
    print(f"Combined PDF files saved in: {pdf_output_dir}")

if __name__ == "__main__":
    main()

Starting JSON file clubbing...


FileNotFoundError: [Errno 2] No such file or directory: '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial_1/same_results/Physics/json'

In [4]:
import os
import json
import csv
import glob

def process_subject_folder(subject_path, subject_name):
    """
    Process a subject folder to extract image names and corresponding JSON data
    
    Args:
        subject_path: Path to the subject folder (Bio, Chem, Maths, Physics)
        subject_name: Name of the subject for CSV filename
    """
    images_folder = os.path.join(subject_path, "images")
    json_folder = os.path.join(subject_path, "json")
    
    # Check if both folders exist
    if not os.path.exists(images_folder) or not os.path.exists(json_folder):
        print(f"Warning: images or json folder not found in {subject_path}")
        return
    
    # Get all image files (assuming common image extensions)
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']
    image_files = []
    for ext in image_extensions:
        image_files.extend(glob.glob(os.path.join(images_folder, ext)))
    
    # Prepare CSV data
    csv_data = []
    csv_headers = ['images', 'json']
    
    for image_file in image_files:
        # Extract image name without extension
        image_name = os.path.splitext(os.path.basename(image_file))[0]
        
        # Look for corresponding JSON file
        json_file_path = os.path.join(json_folder, f"{image_name}.json")
        
        json_content = ""  # Default empty content for second column
        
        if os.path.exists(json_file_path):
            try:
                with open(json_file_path, 'r', encoding='utf-8') as f:
                    json_data = json.load(f)
                
                # Extract the result section if it exists
                if 'result' in json_data:
                    result_data = json_data['result']
                else:
                    result_data = json_data
                
                # Check if JSON has the required structure
                required_fields = [
                    'rotation', 'clarity', 'handwriting', 'writing_style',
                    'lighting', 'resolution', 'paper_condition', 
                    'text_alignment', 'Strikethrough_on_texts'
                ]
                
                # Handle the field name variation (Strikethrough vs Strikethrough_on_texts)
                if 'Strikethrough' in result_data and 'Strikethrough_on_texts' not in result_data:
                    result_data['Strikethrough_on_texts'] = result_data['Strikethrough']
                
                # Verify all required fields exist
                if all(field in result_data for field in required_fields):
                    # Define valid values for each field
                    valid_values = {
                        'rotation': ['upright', 'clockwise', 'anti_clockwise', 'inverted', ''],
                        'clarity': ['slightly_blur', 'very_blur', 'incomprehensible', ''],
                        'handwriting': ['okay', 'bad', 'incomprehensible', ''],
                        'writing_style': ['okay', 'messy', ''],
                        'lighting': ['strong_contrast_shadows', 'glare_on_text', ''],
                        'resolution': ['medium', 'low', 'needs_enhancement', ''],
                        'paper_condition': ['creased', 'torn', 'folded', ''],
                        'text_alignment': ['slightly_misaligned', 'heavily_misaligned', 'overcrowded', ''],
                        'Strikethrough_on_texts': ['present', 'absent', '']
                    }
                    
                    # Check each field and keep only valid ones
                    valid_fields = {}
                    invalid_fields = []
                    
                    for field in required_fields:
                        field_value = result_data.get(field, '')
                        if field_value in valid_values[field]:
                            # Field is valid, keep it
                            valid_fields[field] = field_value
                        else:
                            # Field is invalid, track it for logging
                            invalid_fields.append(f"{field}: '{field_value}'")
                    
                    # Add JSON content if we have any valid fields
                    if valid_fields:
                        # Remove fields with empty values
                        filtered_json = {k: v for k, v in valid_fields.items() if v != ""}
                        if filtered_json:  # Only add if there are non-empty valid fields
                            json_content = json.dumps(filtered_json, indent=1)
                    
                    # Log invalid fields if any
                    if invalid_fields:
                        print(f"Warning: Invalid field values in {json_file_path}: {', '.join(invalid_fields)}")
                else:
                    print(f"Warning: Missing required fields in {json_file_path}")
                    # json_content remains empty
                    
            except json.JSONDecodeError:
                print(f"Warning: Invalid JSON format in {json_file_path}")
                # json_content remains empty
            except Exception as e:
                print(f"Error processing {json_file_path}: {str(e)}")
                # json_content remains empty
        else:
            print(f"Warning: No corresponding JSON file found for {image_name}")
            # json_content remains empty
        
        # Always add the row (with either valid JSON content or empty string)
        csv_data.append([image_name, json_content])
    
    # Write CSV file
    if csv_data:
        csv_filename = f"{subject_name.lower()}_image_quality.csv"
        csv_path = os.path.join(subject_path, csv_filename)
        
        with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(csv_headers)
            writer.writerows(csv_data)
        
        print(f"Created {csv_filename} with {len(csv_data)} rows in {subject_path}")
    else:
        print(f"No valid data found for {subject_name}")

def main():
    """
    Main function to process all subject folders
    """
    # Base path where the subject folders are located
    base_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial/image_quality"
    
    # Subject folders to process
    subjects = ["Bio", "Chem", "Maths", "Physics"]
    
    for subject in subjects:
        subject_path = os.path.join(base_path, subject)
        
        if os.path.exists(subject_path):
            print(f"\nProcessing {subject} folder...")
            process_subject_folder(subject_path, subject)
        else:
            print(f"Warning: {subject} folder not found at {subject_path}")

if __name__ == "__main__":
    main()


In [5]:
import os
import json
import pandas as pd
from pathlib import Path

# Define the base directory
base_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial/image_quality_Final_pro_2.5"

# Define the subjects and their corresponding folder names
subjects = {
    "Bio": "Bio",
    "Chem": "Chem", 
    "Maths": "Maths",
    "Physics": "Physics"
}

# Initialize list to store all data
all_data = []

# Process each subject folder
for subject_key, folder_name in subjects.items():
    json_folder = os.path.join(base_dir, folder_name, "json")
    
    # Check if json folder exists
    if os.path.exists(json_folder):
        # Get all JSON files in the folder
        json_files = [f for f in os.listdir(json_folder) if f.endswith('.json')]
        
        for json_file in json_files:
            json_path = os.path.join(json_folder, json_file)
            
            try:
                # Read and parse JSON file
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                # Extract required information
                row_data = {
                    'image_name': json_file,  # Using JSON filename as requested
                    'subject': subject_key,
                    'rotation': data['result'].get('rotation', ''),
                    'clarity': data['result'].get('clarity', ''),
                    'handwriting': data['result'].get('handwriting', ''),
                    'writing_style': data['result'].get('writing_style', ''),
                    'lighting': data['result'].get('lighting', ''),
                    'resolution': data['result'].get('resolution', ''),
                    'paper_condition': data['result'].get('paper_condition', ''),
                    'text_alignment': data['result'].get('text_alignment', ''),
                    'Strikethrough': data['result'].get('Strikethrough', '')
                }
                
                all_data.append(row_data)
                
            except (json.JSONDecodeError, KeyError, FileNotFoundError) as e:
                print(f"Error processing {json_file}: {e}")
                continue
    else:
        print(f"JSON folder not found for {subject_key}: {json_folder}")

# Create DataFrame
df = pd.DataFrame(all_data)

# Define column order
columns_order = [
    'image_name', 'subject', 'rotation', 'clarity', 'handwriting', 
    'writing_style', 'lighting', 'resolution', 'paper_condition', 
    'text_alignment', 'Strikethrough'
]

# Reorder columns
df = df[columns_order]

# Sort by subject and then by image_name for better organization
df = df.sort_values(['subject', 'image_name']).reset_index(drop=True)

# Save to CSV
output_path = os.path.join(base_dir, "image_quality_pro_2.5.csv")
df.to_csv(output_path, index=False)

print(f"CSV file created successfully: {output_path}")
print(f"Total records processed: {len(df)}")
print(f"Records per subject:")
print(df['subject'].value_counts().sort_index())

# Display first few rows to verify
print("\nFirst 5 rows of the CSV:")
print(df.head())

CSV file created successfully: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_gemini_pcmb/trial/image_quality_Final_pro_2.5/image_quality_pro_2.5.csv
Total records processed: 278
Records per subject:
subject
Bio         28
Chem        50
Maths      138
Physics     62
Name: count, dtype: int64

First 5 rows of the CSV:
                                          image_name subject rotation  \
0   01_1002115268961841141690701450_page_1_1536.json     Bio  upright   
1   02_1002110093815631141684559625_page_1_1536.json     Bio  upright   
2   03_1002102416961841141690700857_page_1_1536.json     Bio  upright   
3   03_1002102416961841141690700857_page_2_1536.json     Bio  upright   
4  04_10021039411060911141694339166_page_1_1536.json     Bio  upright   

         clarity handwriting  writing_style                 lighting  \
0          clear        good  neat_and_tidy               consistent   
1  slightly_blur        okay        